# Weather Disease CatBoost V2 — GPU Training
Notebook này chỉ điều khiển pipeline chung trong `src/models/`; không chứa bản sao logic huấn luyện.

In [ ]:
# Cell 1 — Environment
from pathlib import Path
import sys
import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [start, *start.parents] if (p / 'src/models/training.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Không tìm thấy project root weather_disease_ai_v2')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.common import environment_diagnostics
from src.models.training import GPUTrainingConfig, TrainingPipeline

environment = environment_diagnostics(PROJECT_ROOT)
display(pd.Series(environment, name='value').to_frame())


In [ ]:
# Cell 2 — Configuration (chỉnh toàn bộ cấu hình tại cell duy nhất này)
DEVICE = 'GPU'
GPU_ID = '0'
ITERATIONS = 1000
DEPTH = 8
LEARNING_RATE = 0.05
EARLY_STOPPING = 80

training_config = GPUTrainingConfig(
    device=DEVICE, gpu_id=GPU_ID, iterations=ITERATIONS, depth=DEPTH,
    learning_rate=LEARNING_RATE, early_stopping_rounds=EARLY_STOPPING,
    random_seed=42, verbose=50, allow_writing_files=False,
)
display(pd.Series(training_config.__dict__, name='value').to_frame())


In [ ]:
# Cell 3 — Load và validate data
pipeline = TrainingPipeline(PROJECT_ROOT, training_config)
data = pipeline.validate_data()
display(pipeline.data_summary())
print('Unsupported validation:', data.unsupported_validation)
print('Unsupported test:', data.unsupported_test)


In [ ]:
# Cell 4 — Baseline
baseline = pipeline.run_baseline()
display(pd.DataFrame(baseline['metrics']))


In [ ]:
# Cell 5 — GPU smoke test (dừng notebook nếu thất bại; không fallback CPU)
smoke = pipeline.gpu_smoke_test()
display(pd.Series(smoke, name='value').to_frame())


In [ ]:
# Cell 6 — Train Experiment A
experiment_a = pipeline.train_experiment('current_only')
display(pd.Series(experiment_a).to_frame('value'))


In [ ]:
# Cell 7 — Train Experiment B
experiment_b = pipeline.train_experiment('current_plus_3d')
display(pd.Series(experiment_b).to_frame('value'))


In [ ]:
# Cell 8 — Train Experiment C
experiment_c = pipeline.train_experiment('current_plus_3d_7d')
display(pd.Series(experiment_c).to_frame('value'))


In [ ]:
# Cell 9 — Train Experiment D
experiment_d = pipeline.train_experiment('current_plus_3d_7d_14d')
display(pd.Series(experiment_d).to_frame('value'))


In [ ]:
# Cell 10 — Compare experiments và chọn bằng validation
selected, comparison = pipeline.compare_and_select()
columns = [
    'experiment_label', 'feature_count', 'best_iteration', 'training_seconds',
    'validation_case_weighted_top_1_accuracy', 'validation_case_weighted_top_3_accuracy',
    'validation_case_weighted_top_5_accuracy', 'validation_case_weighted_top_10_accuracy',
    'validation_case_weighted_multiclass_log_loss',
    'validation_case_weighted_macro_f1', 'validation_case_weighted_weighted_f1',
]
display(comparison[columns].sort_values('validation_case_weighted_top_5_accuracy', ascending=False))
print('Selected:', selected['experiment_label'])
print(pipeline.selection_payload['selection_reason'])


In [ ]:
# Cell 11 — Final test (chỉ chạy sau selection)
test_result = pipeline.evaluate_test_once()
display(pd.DataFrame(test_result['metrics']))


In [ ]:
# Cell 12 — Save artifacts
artifact_paths = pipeline.save_artifacts()
display(pd.Series({k: str(v) for k, v in artifact_paths.items()}, name='path').to_frame())


In [ ]:
# Cell 13 — Final verification
verification = pipeline.verify_final_artifacts(sample_rows=5)
display(pd.Series(verification, name='value').to_frame())
assert verification['probabilities_sum_to_one']
assert verification['class_mapping_matches']
